In [102]:
from copy import deepcopy
import lightning as L
import litgpt
from pathlib import Path
import torch

import saws
from warms import ExpCanvas

In [12]:
canvas = ExpCanvas("/work/dlclarge1/mallik-warmstarting/warmstarting_exps/configs/meta_exp_canvas.toml")

In [13]:
BASE_RESULTS_PATH = canvas.results_root

In [14]:
fabric = L.Fabric(devices="auto", strategy="auto")

/work/dlclarge1/mallik-warmstarting/envs/warm_env_global/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /work/dlclarge1/mallik-warmstarting/envs/nyoy_env/l ...


In [15]:
sp_s0_path = BASE_RESULTS_PATH / "talk" / "sp" / "constLR" / "scale0"
sp_s2_path = BASE_RESULTS_PATH / "talk" / "sp" / "constLR" / "scale2"
sp_s4_path = BASE_RESULTS_PATH / "talk" / "sp" / "constLR" / "scale4"

mup_s0_s4_path = BASE_RESULTS_PATH / "talk" / "mup" / "constLR" / "scale0_scale4"
mup_s2_s4_path = BASE_RESULTS_PATH / "talk" / "mup" / "constLR" / "scale2_scale4"

In [16]:
_ckpt_manager = saws.checkpointer.CheckpointManager(
    fabric = fabric,
    load_dir=sp_s0_path,
    save_dir= BASE_RESULTS_PATH / "talk" / "warmstarting" / "debug",
)
sp_s0 = _ckpt_manager._get_last_checkpoint()

In [17]:
_ckpt_manager = saws.checkpointer.CheckpointManager(
    fabric = fabric,
    load_dir=mup_s0_s4_path,
    save_dir= BASE_RESULTS_PATH / "mup_debug" / "warmstarting_trials",
)
mup_s0_s4 = _ckpt_manager._list_available_checkpoints()[0]

In [18]:
base_model = fabric.load(sp_s0)["model"]
target_model = fabric.load(mup_s0_s4)["model"]

/work/dlclarge1/mallik-warmstarting/envs/warm_env_global/lib/python3.10/site-packages/lightning/fabric/utilities/cloud_io.py:57: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [85]:
for i, k in enumerate(base_model.keys()):
    # print(f"{i}\t{k} : {len(base_model[k].shape)} ---> {len(target_model[k].shape)}")
    print(f"{i}\t{k} : {base_model[k].shape} ---> {target_model[k].shape}")

0	lm_head.weight : torch.Size([50688, 48]) ---> torch.Size([50688, 192])
1	transformer.wte.weight : torch.Size([50688, 48]) ---> torch.Size([50688, 192])
2	transformer.h.0.norm_1.weight : torch.Size([48]) ---> torch.Size([192])
3	transformer.h.0.norm_1.bias : torch.Size([48]) ---> torch.Size([192])
4	transformer.h.0.attn.attn.weight : torch.Size([144, 48]) ---> torch.Size([576, 192])
5	transformer.h.0.attn.attn.bias : torch.Size([144]) ---> torch.Size([576])
6	transformer.h.0.attn.proj.weight : torch.Size([48, 48]) ---> torch.Size([192, 192])
7	transformer.h.0.attn.proj.bias : torch.Size([48]) ---> torch.Size([192])
8	transformer.h.0.norm_2.weight : torch.Size([48]) ---> torch.Size([192])
9	transformer.h.0.norm_2.bias : torch.Size([48]) ---> torch.Size([192])
10	transformer.h.0.mlp.fc.weight : torch.Size([192, 48]) ---> torch.Size([768, 192])
11	transformer.h.0.mlp.fc.bias : torch.Size([192]) ---> torch.Size([768])
12	transformer.h.0.mlp.proj.weight : torch.Size([48, 192]) ---> torch.S

In [94]:
import re


def split_architecture(keys):
    """Splits the architecture keys into layers and organizes them.

    Args:
        keys: A list of architecture keys.

    Returns:
        A dictionary where the keys are layer indices and the values are lists of
        parameter names within that layer.
    """

    pattern = r"^(lm_head|transformer\.ln_f|transformer\.wte|transformer\.h\.(\d+)\.(.+))\.([a-z0-9_]+)$"
    layers = {}

    for key in keys:
        match = re.match(pattern, key)
        if match:
            layer_name, layer_index, layer_type, param_type = match.groups()
            if layer_index:
                layer_key = f"transformer.h.{layer_index}"
                if layer_type:
                    layer_key += f".{layer_type}"
                if layer_type:
                    layer_key += f".{param_type}"
                layers.setdefault(layer_key, []).append(key)
            else:
                layers.setdefault(layer_name, []).append(key)

    return layers


# demo
sample_layers = [
    "lm_head.weight",
    "transformer.ln_f.weight",
    "transformer.ln_f.bias",
    "transformer.wte.weight",
    "transformer.h.0.norm_1.weight",
    "transformer.h.0.attn.proj.bias",
    "transformer.h.1.mlp.fc.weight",
]
split_architecture(sample_layers)

{'lm_head': ['lm_head.weight'],
 'transformer.ln_f': ['transformer.ln_f.weight', 'transformer.ln_f.bias'],
 'transformer.wte': ['transformer.wte.weight'],
 'transformer.h.0.norm_1.weight': ['transformer.h.0.norm_1.weight'],
 'transformer.h.0.attn.proj.bias': ['transformer.h.0.attn.proj.bias'],
 'transformer.h.1.mlp.fc.weight': ['transformer.h.1.mlp.fc.weight']}

In [82]:
count = 0
for k, v in split_architecture(base_model.keys()).items():
    count += len(v)

# verifying parsing of architecture
print(len(base_model), count, len(base_model) == count)

76 76 True


In [119]:
def _pad_zeros_to_tensor(base: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Pads the base tensor with zeros to match the shape of the target tensor.
    
    Args:
        base: The tensor to be padded.
        target: The tensor to match the shape of.

    Returns:
        The padded tensor.
    """
    assert len(base.shape) == len(target.shape), "Tensors must have the same number of dimensions."

    if base.shape == target.shape:
        # copies the base model weights to the target model as the sizes are the same
        return base
    
    assert all(t >= b for t, b in zip(target.shape, base.shape)), \
        "The target tensor must be greater or equal to the base tensor for every dimension."

    # create a target-sized tensor with 0s
    target = torch.zeros_like(target)
    # add the base matrix to the target matrix from the top-left corner
    ## the below operation is equivalent to:- target[:base.shape[0], :base.shape[1]] = base
    ## but generalized for n-dimensional tensors
    target[tuple(slice(0, dim) for dim in base.shape)] = base
    return target

def pad_zeros(
    base: torch.nn.Module | dict,
    target: torch.nn.Module | dict,
) -> torch.nn.Module | dict:
    """ Pads the base model with zeros to match the shape of the target model.

    Args:
        base: The base model.
        target: The target model.

    Returns:
        The padded base model.
    """
    if isinstance(base, torch.nn.Module):
        base = base.state_dict()
    _target = deepcopy(target) 
    if isinstance(target, torch.nn.Module):
        _target = target.state_dict()

    assert sorted(base.keys()) == sorted(_target.keys()), \
        "The keys of the base and target models must match."

    for k in _target.keys():
        _target[k] = _pad_zeros_to_tensor(base[k], _target[k])

    if isinstance(target, torch.nn.Module):
        target.load_state_dict(_target)
    else:
        target = _target
    return target


In [133]:
_t = pad_zeros(base_model, target_model)

In [135]:
k = "transformer.h.3.attn.attn.weight"
print(base_model[k].shape, '\n', base_model[k])
print(_t[k].shape, '\n', _t[k])

torch.Size([144, 48]) 
 tensor([[ 0.5603, -0.8841, -0.2198,  ...,  0.7899, -0.0325, -0.6522],
        [ 0.3670,  0.8994, -0.2831,  ...,  1.0953, -1.5427,  0.1471],
        [ 0.0700, -1.1179, -0.5939,  ..., -1.5938,  1.0363,  0.2998],
        ...,
        [ 0.9259, -0.1093, -0.8449,  ...,  0.2481,  0.4990, -0.9341],
        [ 0.9296, -0.2219, -0.0936,  ..., -0.2587, -0.1481,  0.1432],
        [-1.1537, -0.0646,  0.0559,  ...,  0.8069,  0.6487, -0.6681]])
torch.Size([576, 192]) 
 tensor([[ 0.5603, -0.8841, -0.2198,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.3670,  0.8994, -0.2831,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0700, -1.1179, -0.5939,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])
